# Study 3b

- Upbringing (Good/Bad) × Action (Help/Harm)
- Blame (+1 to +7), Praise (+1 to +7), True Self (+1 to +7)
- N = 276

## Set Up Notebook

In [ ]:
##########
# IMPORT #
##########

# Data processing and math
import pandas as pd
import numpy as np

# Statistics
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Display handling
import warnings


#################
# CONFIGURATION #
#################

# Suppress warnings
warnings.filterwarnings('ignore')

# Set global plotting style
sns.set_theme(style = "whitegrid")


##########
# LABELS #
##########
labels_gender = {
    '1': 'Female',
    '2': 'Male',
    '3': 'Non-binary',
    '4': 'Prefer not to say'
}


#################
# VISUALIZATION #
#################
palette_main = {
    "Good": "#0072B2",
    "Bad":  "#D55E00"
}

## Transform Data

In [ ]:
# Load data
df_clean = pd.read_csv('blame_praise_self_study_3b_data.csv', sep = ';')

# Define factors
factors_map = {
    'non,blameAc':         {'Upbringing': 'Good', 'Action': 'Harm (Blame)',  'Measure': 'Blame or Praise'},
    'non,blameSelf':       {'Upbringing': 'Good', 'Action': 'Harm (Blame)',  'Measure': 'True Self'},
    'non,praiseAc':        {'Upbringing': 'Good', 'Action': 'Help (Praise)', 'Measure': 'Blame or Praise'},
    'non,praiseSelf':      {'Upbringing': 'Good', 'Action': 'Help (Praise)', 'Measure': 'True Self'},
    'ignorant,blame,AC':   {'Upbringing': 'Bad',  'Action': 'Harm (Blame)',  'Measure': 'Blame or Praise'},
    'ignorant,blameSelf':  {'Upbringing': 'Bad',  'Action': 'Harm (Blame)',  'Measure': 'True Self'},
    'ignorant,praiseAc':   {'Upbringing': 'Bad',  'Action': 'Help (Praise)', 'Measure': 'Blame or Praise'},
    'ignorant,praiseSelf': {'Upbringing': 'Bad',  'Action': 'Help (Praise)', 'Measure': 'True Self'}
}

# Extract leading digit as integer
def extract_score(val):
    if pd.isna(val): return np.nan
    s = str(val)[0]
    return int(s) if s.isdigit() else np.nan

# Reshape wide to long
long_data = []

for _, row in df_clean.iterrows():
    for col, factors in factors_map.items():
        score = extract_score(row.get(col))
        if not np.isnan(score):
            entry = {'ID': row['ID'], 'Score': score}
            entry.update(factors)
            long_data.append(entry)

df_long = pd.DataFrame(long_data)

print(f"Transformation complete ({len(df_long)} Observations).")

## Calculate Sociodemographics

In [ ]:
# Prepare data
df_clean['Gender_Labeled'] = df_clean['Gender'].astype(str).str.replace('.0', '', regex = False).map(labels_gender)
df_clean['Age'] = pd.to_numeric(df_clean['Age'], errors = 'coerce')

# Display results
for col in ['Gender_Labeled']:
    print(f"\n{col}:")
    display(df_clean[col].value_counts().to_frame('Frequency'))

print("\nAge:")
display(df_clean['Age'].describe().to_frame('Value').round(3))

print("\nParticipants:")
total_participants = df_long['ID'].nunique()
display(f"N = {total_participants}")

## Calculate Descriptive Statistics

In [ ]:
# Calculate descriptive statistics
descriptive_stats = df_long.groupby(['Measure', 'Action', 'Upbringing'])['Score'].agg(['mean', 'std', 'count']).round(3)

# Display results
display(descriptive_stats)

## Perform t-Tests

In [ ]:
# Initialize list
t_results = []

# Iterate through measures
for measure in df_long['Measure'].unique():
    
    # Iterate through actions
    for action in df_long['Action'].unique():
        
        # Filter by upbringing
        good_scores = df_long[(df_long['Measure']    == measure) &
                              (df_long['Action']     == action)  &
                              (df_long['Upbringing'] == 'Good')]['Score']
        
        bad_scores  = df_long[(df_long['Measure']    == measure) &
                              (df_long['Action']     == action)  &
                              (df_long['Upbringing'] == 'Bad')]['Score']
        
        # Run Welch's t-test
        t_stat, p_val = stats.ttest_ind(good_scores, bad_scores, equal_var = False)
        
        # Store results
        t_results.append({
            'DV': measure,
            'Action': action,
            'Mean Good Upbr.': round(good_scores.mean(), 3),
            'Mean Bad Upbr.': round(bad_scores.mean(), 3),
            't': round(t_stat, 3),
            'p': round(p_val, 3)
        })

# Display results
df_t_results = pd.DataFrame(t_results)
display(df_t_results)

## Generate Histograms

In [ ]:
# Create condition column
df_long['Condition'] = df_long['Upbringing'] + " / " + df_long['Action']

# Iterate through measures
for measure in df_long['Measure'].unique():
    
    # Filter by measure
    subset = df_long[df_long['Measure'] == measure]
    
    # Generate histograms
    g = sns.displot(data     = subset,
                    x        = 'Score',
                    col      = "Condition",
                    col_wrap = 4,
                    kind     = "hist",
                    discrete = True,
                    shrink   = 0.8,
                    height   = 3,
                    aspect   = 1.2,
                    color    = "#0072B2"
                   )
    
    # Set labels and titles
    g.set_axis_labels("Score", "Count")
    g.set_titles(f"{measure}\n{{col_name}}")
    
    # Format axes
    for ax in g.axes.flat:
        ax.set_xticks(range(1, 8))
        ax.set_xlim(0.5, 7.5)
    
    # Export graph
    clean_name = measure.replace("/", "or").replace(" ", "_").lower()
    filename = f'blame_praise_self_study_3b_hist_{clean_name}.png'
    g.savefig(filename, dpi = 300, bbox_inches = 'tight')
    print(f"Graph saved as '{filename}'.")
    plt.show()

## Generate Bar Plots

In [ ]:
# Iterate through measures
for measure in df_long['Measure'].unique():
    
    # Set figure size
    plt.figure(figsize = (8, 5))
    
    # Generate graph
    g = sns.barplot(data    = df_long[df_long['Measure'] == measure],
                    x       = "Action",
                    y       = "Score",
                    hue     = "Upbringing",
                    palette = palette_main,
                    capsize = .05,
                    errorbar = "se"
                   )
    
    # Set axis labels and titles
    plt.ylabel('Mean Score', fontsize = 12)
    plt.xlabel('Action', fontsize = 12)
    
    # Set y-axis limit
    plt.ylim(1, 7.5)
    
    # Set main title
    plt.title(f'{measure}', fontsize = 14)
    
    # Set legend
    plt.legend(title = 'Upbringing', bbox_to_anchor = (1.05, 1), loc = 'upper left')
    
    # Export graph
    # clean_name = measure.replace("/", "or").replace(" ", "_").lower()
    # filename = f'blame_praise_self_study_3b_{clean_name}.png'
    # plt.savefig(filename, dpi = 300, bbox_inches = 'tight')
    plt.show()
    # print(f"Graph saved as '{filename}'.")